# Telco Customer Churn Prediction

**Educational project.** The data describes a *fictional* telecom company (IBM sample data). Nothing here is validated on real operational data or ready for production.

**Goal:** predict which customers are likely to leave (`Churn = Yes`) so a retention team can prioritise outreach.

**Workflow:** load -> inspect -> stratified split -> EDA (training data only) -> feature engineering -> model comparison -> tuning -> feature selection -> threshold + calibration -> one final test evaluation -> save model for Streamlit.

## 1. Business Problem and Primary Metric (defined before any modelling)

- **False negative** = a customer who leaves but is not flagged -> lost revenue, no chance to retain.
- **False positive** = a loyal customer flagged -> wasted discount or call.

**Primary metric: F1-score of the churn class** (balances precision and recall; higher is better). It is a sensible default when we do not know the retention budget.
**Secondary metrics:** precision, recall, ROC-AUC and **Average Precision (PR-AUC)**, which shows how well customers are *ranked* by risk and is more informative than accuracy under class imbalance. Accuracy is not used to select models.

If the retention team had a fixed capacity (e.g. 500 calls per month) we would instead optimise precision at that capacity; that information is not available here.

## 2. Dataset Source and License

- **Data file:** https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
- **Repository:** https://github.com/IBM/telco-customer-churn-on-icp4d (archived by IBM in July 2024; code is Apache-2.0)
- **Original context:** IBM Cognos Analytics sample data "Telco customer churn" (fictional telco).
- **License caveat:** the repository's Apache-2.0 license covers the *code pattern*. I could not find an explicit license for the CSV itself, so it is **not** copied into this repo; the notebook downloads it, and `data/README.md` explains how to get it. Credit IBM as the source.

## 3. Imports and Configuration

In [ ]:
import os
import sys
import json
import warnings

# Make churn_utils.py (repo root) importable, also inside parallel worker processes
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_predict, RandomizedSearchCV, GridSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, brier_score_loss, log_loss,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

from churn_utils import SEED, FEATURE_SETS, add_features, build_pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 4. Data Loading

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
LOCAL_PATH = os.path.join(REPO_ROOT, "data", "Telco-Customer-Churn.csv")

try:
    df_raw = pd.read_csv(DATA_URL)
    print("Loaded from URL")
except Exception as error:
    print("URL not reachable:", type(error).__name__, "- trying local file", LOCAL_PATH)
    df_raw = pd.read_csv(LOCAL_PATH)   # download the CSV manually into data/ (see data/README.md)

## 5. Initial Data Inspection

Only structure is checked here; no statistics are learned yet. The commonly documented size is 7,043 x 21; we verify it rather than assume it.

In [ ]:
print("Rows, columns:", df_raw.shape, "(documented: (7043, 21))")
display(df_raw.head(3))
display(df_raw.dtypes.to_frame("dtype").T)

print("\nMissing values (NaN) per column:", int(df_raw.isna().sum().sum()), "in total")
print("Duplicated customerID:", df_raw["customerID"].duplicated().sum())
print("Fully duplicated rows ignoring customerID:", df_raw.drop(columns="customerID").duplicated().sum())

# Unique values of the text columns (spot typos / odd categories)
for col in df_raw.select_dtypes(include=["object", "string"]).columns.drop(["customerID", "TotalCharges"]):
    print(f"{col:18s}", sorted(df_raw[col].unique()))

### 5.1 `TotalCharges` investigation

`TotalCharges` is read as text, which suggests blank values. Hypothesis: blanks belong to customers with `tenure == 0` (not billed yet). We test it before choosing a fix.

In [ ]:
total_text = df_raw["TotalCharges"].astype(str).str.strip()
blank_mask = total_text == ""
print("Blank TotalCharges:", blank_mask.sum())
print("Tenure values of those rows:", df_raw.loc[blank_mask, "tenure"].unique())
print("Rows with tenure == 0:", (df_raw["tenure"] == 0).sum())
print("Do blanks and tenure==0 coincide exactly?", bool((blank_mask == (df_raw["tenure"] == 0)).all()))
print("Churn label of the blank rows:", df_raw.loc[blank_mask, "Churn"].value_counts().to_dict())

**Decision rule:** if the blanks match `tenure == 0` exactly, they are *structural zeros* (no billing yet), so `add_features()` sets them to 0. This is a fixed business rule, not a statistic learned from data, so doing it before the split cannot leak. A median imputer stays in the pipeline as a safety net for any other missing value.

## 6. Target and Leakage Review

- Target `Churn`: `Yes` -> 1 (left), `No` -> 0 (stayed).
- `customerID` is an identifier with no predictive meaning -> dropped. It is unique, so the same customer cannot appear in both train and test.
- All other columns are demographics, services, contract and charges. **Limitation:** the file has no timestamps, so I cannot prove every column was known *before* the customer left. In particular `tenure` and `TotalCharges` of churners end at the leaving month. A real deployment would need features snapshotted before the churn event.

In [ ]:
y = (df_raw["Churn"] == "Yes").astype(int)
X = df_raw.drop(columns=["customerID", "Churn"])
print("Features:", X.shape[1], "| target mean (overall churn rate):", round(y.mean(), 4))

## 7. Train-Test Split

Stratified 80/20 split with a fixed seed **before** EDA and any fitting, so the test set cannot influence decisions. Model selection uses 5-fold stratified CV on the training part only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Train:", X_train.shape, "churn rate", round(y_train.mean(), 4))
print("Test :", X_test.shape, "churn rate", round(y_test.mean(), 4), "(kept untouched until the end)")

## 8. Exploratory Data Analysis (training data only)

In [ ]:
eda_df = add_features(X_train).assign(Churn=y_train)

# Q: how imbalanced is the target?
counts = eda_df["Churn"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["Stayed (0)", "Churned (1)"], counts.values, color=["#4c72b0", "#c44e52"])
ax.set_title("Target distribution (train)")
plt.show()
print(f"Churn rate: {y_train.mean():.1%}  | imbalance ratio stayed:churned = {counts[0] / counts[1]:.2f}:1")

**Why:** the imbalance decides the metric (F1/AP instead of accuracy) and whether we need class weights. A moderate imbalance is handled with `class_weight="balanced"` in all models, plus threshold tuning later. No resampling (SMOTE) is used: it adds complexity and risks leakage if done outside CV.

In [ ]:
# Q: which categorical features separate churners from stayers?
cat_to_plot = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "TechSupport", "OnlineSecurity"]
overall = eda_df["Churn"].mean()
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), cat_to_plot):
    rate = eda_df.groupby(col)["Churn"].mean().sort_values()
    rate.plot.barh(ax=ax, color="#4c72b0")
    ax.axvline(overall, color="red", linestyle="--", label="overall")
    ax.set_title(f"Churn rate by {col}")
    ax.set_xlabel("churn rate")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

**Why:** large gaps against the red line mean the feature carries signal -> keep and one-hot encode it. Nominal columns use one-hot encoding (no false ordering).

In [ ]:
# Q: how do tenure and charges differ between churners and stayers? Are they skewed / do they have outliers?
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for i, col in enumerate(num_cols):
    sns.histplot(data=eda_df, x=col, hue="Churn", stat="density", common_norm=False, kde=True, ax=axes[0, i])
    sns.boxplot(data=eda_df, x="Churn", y=col, ax=axes[1, i])
plt.tight_layout()
plt.show()

print("Skewness:\n", eda_df[num_cols].skew().round(2).to_string())
for col in num_cols:
    q1, q3 = eda_df[col].quantile([0.25, 0.75])
    n_out = ((eda_df[col] < q1 - 1.5 * (q3 - q1)) | (eda_df[col] > q3 + 1.5 * (q3 - q1))).sum()
    print(f"{col}: {n_out} IQR outliers")

**Why:** outliers here are real customers (long tenures, high bills), not errors, so none are removed. Logistic Regression gets standard scaling; tree models do not need it. A log transform would be justified only if skewness were strong; check the printed values.

In [ ]:
# Q: do engineered ideas look useful? (service count, tenure band, autopay)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
eda_df.groupby("tenure_band")["Churn"].mean().reindex(["0-12", "13-24", "25-48", "49+"]).plot.bar(ax=axes[0], color="#4c72b0")
axes[0].set_title("Churn rate by tenure band")
eda_df.groupby("n_services")["Churn"].mean().plot.bar(ax=axes[1], color="#4c72b0")
axes[1].set_title("Churn rate by number of services")
eda_df.groupby("autopay")["Churn"].mean().plot.bar(ax=axes[2], color="#4c72b0")
axes[2].set_title("Churn rate by autopay (1 = yes)")
plt.tight_layout()
plt.show()

In [ ]:
# Q: are TotalCharges, MonthlyCharges and tenure redundant?
eda_df["tenure_x_monthly"] = eda_df["tenure"] * eda_df["MonthlyCharges"]
print("corr(TotalCharges, tenure)            :", round(eda_df["TotalCharges"].corr(eda_df["tenure"]), 3))
print("corr(TotalCharges, tenure*Monthly)    :", round(eda_df["TotalCharges"].corr(eda_df["tenure_x_monthly"]), 3))

corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "avg_monthly_spend", "n_services", "autopay", "SeniorCitizen", "Churn"]
plt.figure(figsize=(8, 6))
sns.heatmap(eda_df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (train)")
plt.show()

**Why:** `TotalCharges` is roughly tenure x monthly price, so it is expected to be strongly correlated with `tenure`. That is multicollinearity, which matters for Logistic Regression coefficients but not much for trees. We do **not** drop it by assumption; section 11 compares feature sets with and without it.

## 9. Data Quality Assessment

In [ ]:
quality = pd.DataFrame({
    "missing_raw": X_train.isna().sum(),
    "n_unique": X_train.nunique(),
})
display(quality.T)

dups = X_train.duplicated().sum()
print("Duplicate feature rows in train:", dups, "(different customers can legitimately share all attributes; kept)")

# Rare categories (< 1% of rows) could cause unstable one-hot columns
for col in X_train.select_dtypes(include=["object", "string"]).columns.drop("TotalCharges"):
    share = X_train[col].value_counts(normalize=True)
    if share.min() < 0.01:
        print("Rare category in", col, share[share < 0.01].to_dict())
print("Rare-category check finished. OneHotEncoder(handle_unknown='ignore') protects the app against unseen values.")

**Summary of cleaning decisions**

| Change | Why | Evidence | Leakage risk |
|---|---|---|---|
| Drop `customerID` | identifier | unique per row | none |
| `TotalCharges` text -> float, blanks -> 0 | structural zeros | section 5.1 | none (fixed rule) |
| Median imputer in pipeline | safety net | fitted per training fold | none |
| One-hot encode nominal columns | no natural order | EDA | encoder fitted per fold |
| Keep outliers | real customers | boxplots | none |
| No resampling | `class_weight` is simpler and CV-safe | imbalance moderate | none |

## 10. Feature Engineering

All engineered features use only the *same row*, so they can be computed for a new customer in the app and cannot leak information from other rows or the target.

| Feature | Definition | Business reasoning |
|---|---|---|
| `avg_monthly_spend` | `TotalCharges / tenure` (MonthlyCharges if tenure = 0) | captures price changes over time vs. current `MonthlyCharges` |
| `n_services` | count of "Yes" among 8 service columns | engagement / stickiness |
| `tenure_band` | 0-12, 13-24, 25-48, 49+ months | non-linear tenure effect for linear models |
| `autopay` | payment method is automatic | convenience may reduce churn |

Features are tested, not assumed: next section compares **original**, **original without TotalCharges**, and **engineered** sets with identical CV folds.

In [ ]:
display(add_features(X_train)[["tenure", "MonthlyCharges", "TotalCharges", "avg_monthly_spend", "n_services", "tenure_band", "autopay"]].head())

## 11. Models, Evaluation Function and Feature-Set Comparison

Models: **Logistic Regression** (interpretable baseline), **Random Forest** (non-linear, interactions), **HistGradientBoosting** (strong on tabular data, native to scikit-learn). All use `class_weight="balanced"`. Logistic Regression gets scaling; trees do not.
A stratified random classifier gives the "chance" reference.

In [ ]:
def make_model(name):
    """Return an unfitted model with sensible defaults."""
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced",
                                      random_state=SEED, n_jobs=1)
    if name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED)
    if name == "Random baseline":
        return DummyClassifier(strategy="stratified", random_state=SEED)
    raise ValueError(name)


def make_pipeline(name, feature_set, select=False):
    """Pipeline for one model and one feature set (scaling only for Logistic Regression)."""
    numeric_cols, categorical_cols = FEATURE_SETS[feature_set]
    return build_pipeline(make_model(name), numeric_cols, categorical_cols,
                          scale=(name == "Logistic Regression"), select=select)


SCORING = {"f1": "f1", "precision": "precision", "recall": "recall",
           "roc_auc": "roc_auc", "avg_precision": "average_precision"}


def evaluate_pipeline(label, pipe, X, y):
    """5-fold CV on training data. Returns train/validation scores and gaps in one row."""
    s = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, return_train_score=True, n_jobs=-1)
    return {
        "model": label,
        "train_f1": s["train_f1"].mean(),
        "val_f1": s["test_f1"].mean(),
        "val_f1_std": s["test_f1"].std(),
        "f1_gap": s["train_f1"].mean() - s["test_f1"].mean(),
        "val_precision": s["test_precision"].mean(),
        "val_recall": s["test_recall"].mean(),
        "val_roc_auc": s["test_roc_auc"].mean(),
        "val_avg_precision": s["test_avg_precision"].mean(),
        "train_avg_precision": s["train_avg_precision"].mean(),
        "ap_gap": s["train_avg_precision"].mean() - s["test_avg_precision"].mean(),
    }


MODEL_NAMES = ["Logistic Regression", "Random Forest", "HistGradientBoosting"]
rows = []
for feature_set in FEATURE_SETS:
    for name in MODEL_NAMES:
        row = evaluate_pipeline(name, make_pipeline(name, feature_set), X_train, y_train)
        row["feature_set"] = feature_set
        rows.append(row)
fs_results = pd.DataFrame(rows)

print("Validation F1 (mean of 5 folds) by feature set:")
display(fs_results.pivot(index="model", columns="feature_set", values="val_f1").round(4))
print("Validation Average Precision by feature set:")
display(fs_results.pivot(index="model", columns="feature_set", values="val_avg_precision").round(4))

**Decision rule (set before looking at the numbers):** pick the feature set with the best mean validation F1 across the three models, but only switch away from `original` if it wins by at least 0.005 (about the noise level of CV); otherwise prefer the simpler set.

In [ ]:
mean_f1 = fs_results.groupby("feature_set")["val_f1"].mean()
print(mean_f1.round(4).to_string())
best_fs = mean_f1.idxmax()
if mean_f1[best_fs] - mean_f1["original"] < 0.005:
    best_fs = "original"
print("\nSelected feature set:", best_fs)

## 12. Model Comparison (selected feature set, default hyperparameters)

In [ ]:
baseline_row = evaluate_pipeline("Random baseline", make_pipeline("Random baseline", best_fs), X_train, y_train)
baseline_row["feature_set"] = best_fs
comparison = pd.concat([pd.DataFrame([baseline_row]), fs_results[fs_results["feature_set"] == best_fs]], ignore_index=True)
display(comparison.drop(columns="feature_set").round(4))

**How to read it:** `f1_gap` = train F1 - validation F1. A large gap hints at overfitting (typical for Random Forest, which fits the training data almost perfectly); a small gap with a low score hints at underfitting. Compare gaps together with the fold standard deviation `val_f1_std`; a difference smaller than the std is not meaningful.

## 13. Hyperparameter Tuning

Optimised metric: **F1** with the same 5 folds. Small search spaces keep it cheap (grid for Logistic Regression, 15 random draws for the tree models).

| Model | Tuned parameters | Why they matter |
|---|---|---|
| Logistic Regression | `C` | regularisation strength (collinear features) |
| Random Forest | `max_depth`, `min_samples_leaf`, `max_features` | control tree complexity / overfitting |
| HistGradientBoosting | `learning_rate`, `max_iter`, `max_depth`, `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization` | trade-off between fit and regularisation |

In [ ]:
param_spaces = {
    "Logistic Regression": {"model__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    "Random Forest": {"model__max_depth": [4, 6, 8, 12, None],
                      "model__min_samples_leaf": [1, 3, 5, 10, 20],
                      "model__max_features": ["sqrt", 0.3, 0.5]},
    "HistGradientBoosting": {"model__learning_rate": [0.02, 0.05, 0.1],
                             "model__max_iter": [100, 200, 300],
                             "model__max_depth": [2, 3, 4, None],
                             "model__max_leaf_nodes": [8, 15, 31],
                             "model__min_samples_leaf": [20, 40, 80],
                             "model__l2_regularization": [0, 1, 10]},
}

searches = {}
for name in MODEL_NAMES:
    base = make_pipeline(name, best_fs)
    if name == "Logistic Regression":
        search = GridSearchCV(base, param_spaces[name], scoring="f1", cv=cv, n_jobs=-1)
    else:
        search = RandomizedSearchCV(base, param_spaces[name], n_iter=15, scoring="f1", cv=cv,
                                    random_state=SEED, n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name}: best CV F1 = {search.best_score_:.4f} | {search.best_params_}")

In [ ]:
tuned_rows = []
for name in MODEL_NAMES:
    row = evaluate_pipeline(name + " (tuned)", searches[name].best_estimator_, X_train, y_train)
    row["f1_before_tuning"] = comparison.loc[comparison["model"] == name, "val_f1"].iloc[0]
    row["f1_change"] = row["val_f1"] - row["f1_before_tuning"]
    tuned_rows.append(row)
tuned = pd.DataFrame(tuned_rows)
display(tuned[["model", "train_f1", "val_f1", "val_f1_std", "f1_gap", "val_precision", "val_recall",
               "val_roc_auc", "val_avg_precision", "f1_before_tuning", "f1_change"]].round(4))

final_name = tuned.loc[tuned["val_f1"].idxmax(), "model"].replace(" (tuned)", "")
best_params = searches[final_name].best_params_
print("Best model by validation F1:", final_name, best_params)
print("Note: tuning and this evaluation reuse the same folds, so these scores are slightly optimistic; the test set gives the unbiased check.")

## 14. Feature Importance

- **Permutation importance** on a validation part held out *inside the training set*; it works on the original columns, so one-hot columns of the same feature are not split apart. It is computed with Average Precision (more stable than F1).
- **Logistic Regression coefficients** as a second, interpretable view (scaled numeric features; one-hot columns).

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=SEED)
importance_pipe = clone(searches[final_name].best_estimator_).fit(X_fit, y_fit)
perm = permutation_importance(importance_pipe, X_val, y_val, scoring="average_precision",
                              n_repeats=10, random_state=SEED, n_jobs=1)
perm_df = (pd.DataFrame({"feature": X_val.columns, "mean": perm.importances_mean, "std": perm.importances_std})
           .sort_values("mean", ascending=True))
perm_df.tail(15).plot.barh(x="feature", y="mean", xerr=perm_df.tail(15)["std"], legend=False, figsize=(7, 6), color="#4c72b0")
plt.title(f"Permutation importance ({final_name}, drop in Average Precision)")
plt.show()

In [ ]:
lr_pipe = clone(searches["Logistic Regression"].best_estimator_).fit(X_train, y_train)
lr_names = lr_pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=lr_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index)[:15].sort_values()
top.plot.barh(figsize=(7, 6), color=["#c44e52" if v > 0 else "#4c72b0" for v in top])
plt.title("Logistic Regression: largest coefficients (red = higher churn risk)")
plt.show()

**Caution:** correlated features (e.g. `tenure`, `TotalCharges`, `avg_monthly_spend`) share importance, so each looks smaller than their combined effect. Importance is not causation.

## 15. Feature Selection: Full vs Reduced Model

The reduced model adds an **L1-regularised linear SVM selector (`LinearSVC`)** inside the pipeline, so selection is re-fitted on each training fold (no leakage) and also applies automatically to raw app inputs. We actually train and cross-validate both models with the same folds.

**Rule:** use the reduced model if its validation F1 is within 0.005 of the full model *and* it keeps fewer features (simpler, easier to explain).

In [ ]:
full_pipe = clone(searches[final_name].best_estimator_)
reduced_pipe = make_pipeline(final_name, best_fs, select=True).set_params(**best_params)

full_row = evaluate_pipeline("Full features", full_pipe, X_train, y_train)
reduced_row = evaluate_pipeline("Reduced (L1 selection)", reduced_pipe, X_train, y_train)

# How many one-hot/numeric columns are kept when fitted on the whole training set?
fitted_reduced = clone(reduced_pipe).fit(X_train, y_train)
all_names = fitted_reduced.named_steps["prep"].get_feature_names_out()
kept_mask = fitted_reduced.named_steps["select"].get_support()
full_row["n_features"], reduced_row["n_features"] = len(all_names), int(kept_mask.sum())

display(pd.DataFrame([full_row, reduced_row])[["model", "n_features", "train_f1", "val_f1", "val_f1_std", "f1_gap",
                                              "val_avg_precision", "val_roc_auc"]].round(4))
print("Dropped by the selector:", list(all_names[~kept_mask]))

use_reduced = (reduced_row["val_f1"] >= full_row["val_f1"] - 0.005) and (reduced_row["n_features"] < full_row["n_features"])
final_pipe = reduced_pipe if use_reduced else full_pipe
print("\nUse reduced model:", use_reduced)

## 16. Probability Calibration and Threshold Optimisation (training data only)

Out-of-fold (OOF) probabilities from cross-validation are used, so every customer is scored by a model that never saw them. We compare:

1. **uncalibrated** probabilities (class weights push them upward),
2. **sigmoid** and **isotonic** calibration (`CalibratedClassifierCV`, fitted inside each fold).

Selection rule: lowest **Brier score**; keep the uncalibrated model unless calibration improves Brier by at least 0.001. The **threshold** is then chosen on the selected variant's OOF probabilities by maximising **F1** (the predefined primary metric). The test set is not used.

In [ ]:
candidates = {
    "uncalibrated": final_pipe,
    "sigmoid": CalibratedClassifierCV(final_pipe, method="sigmoid", cv=5),
    "isotonic": CalibratedClassifierCV(final_pipe, method="isotonic", cv=5),
}
thresholds = np.arange(0.05, 0.951, 0.01)


def threshold_table(y_true, proba):
    """Precision, recall and F1 for each threshold in the grid."""
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        rows.append({"threshold": round(t, 2),
                     "precision": precision_score(y_true, pred, zero_division=0),
                     "recall": recall_score(y_true, pred, zero_division=0),
                     "f1": f1_score(y_true, pred, zero_division=0)})
    return pd.DataFrame(rows)


oof, summary_rows = {}, []
for label, est in candidates.items():
    oof[label] = cross_val_predict(est, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    tt = threshold_table(y_train, oof[label])
    best = tt.loc[tt["f1"].idxmax()]
    summary_rows.append({"variant": label,
                         "brier": brier_score_loss(y_train, oof[label]),
                         "log_loss": log_loss(y_train, oof[label]),
                         "avg_precision": average_precision_score(y_train, oof[label]),
                         "roc_auc": roc_auc_score(y_train, oof[label]),
                         "best_threshold": best["threshold"], "f1_at_best": best["f1"]})
calib_summary = pd.DataFrame(summary_rows).set_index("variant")
display(calib_summary.round(4))

best_label = calib_summary["brier"].idxmin()
if calib_summary.loc["uncalibrated", "brier"] - calib_summary.loc[best_label, "brier"] < 0.001:
    best_label = "uncalibrated"
print("Selected variant:", best_label)

plt.figure(figsize=(5, 5))
for label in candidates:
    frac_pos, mean_pred = calibration_curve(y_train, oof[label], n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", label=label)
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed churn rate")
plt.title("Reliability curve (OOF, train)")
plt.legend()
plt.show()

In [ ]:
thr_table = threshold_table(y_train, oof[best_label])
best_threshold = float(thr_table.loc[thr_table["f1"].idxmax(), "threshold"])

plt.figure(figsize=(7, 4))
for metric in ["precision", "recall", "f1"]:
    plt.plot(thr_table["threshold"], thr_table[metric], label=metric)
plt.axvline(0.5, color="grey", linestyle=":", label="default 0.5")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"chosen {best_threshold:.2f}")
plt.xlabel("Threshold")
plt.legend()
plt.title("Threshold vs metrics (OOF, train)")
plt.show()

compare_thr = thr_table[thr_table["threshold"].isin([0.5, best_threshold])].drop_duplicates("threshold")
display(compare_thr.round(4))
print(f"Chosen threshold: {best_threshold:.2f}")

**Operational meaning:** a *lower* threshold flags more customers (higher recall, lower precision, bigger retention workload); a *higher* one flags fewer but more certain customers. F1 weights both equally. If a retention call costs much less than a lost customer, a lower threshold than the F1-optimal one may be better; the plot above shows the trade-off.

## 17. Final Evaluation on the Untouched Test Set

Everything is now fixed: feature set, model, hyperparameters, feature selection, calibration and threshold. The model is refit on all training data and evaluated **once** on the test set.

In [ ]:
final_model = candidates[best_label]
final_model.fit(X_train, y_train)

proba_test = final_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_threshold).astype(int)
proba_train = final_model.predict_proba(X_train)[:, 1]

test_metrics = {
    "f1": f1_score(y_test, pred_test),
    "precision": precision_score(y_test, pred_test),
    "recall": recall_score(y_test, pred_test),
    "avg_precision": average_precision_score(y_test, proba_test),
    "roc_auc": roc_auc_score(y_test, proba_test),
    "brier": brier_score_loss(y_test, proba_test),
}
print({k: round(v, 4) for k, v in test_metrics.items()})

# Train vs validation (OOF) vs test, all at the chosen threshold
oof_pred = (oof[best_label] >= best_threshold).astype(int)
train_pred = (proba_train >= best_threshold).astype(int)
display(pd.DataFrame({
    "train": [f1_score(y_train, train_pred), precision_score(y_train, train_pred), recall_score(y_train, train_pred)],
    "validation (OOF)": [f1_score(y_train, oof_pred), precision_score(y_train, oof_pred), recall_score(y_train, oof_pred)],
    "test": [test_metrics["f1"], test_metrics["precision"], test_metrics["recall"]],
}, index=["F1", "precision", "recall"]).round(4))

print(classification_report(y_test, pred_test, target_names=["Stayed", "Churned"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, display_labels=["Stayed", "Churned"], cmap="Blues")
plt.title(f"Test confusion matrix (threshold {best_threshold:.2f})")
plt.show()

# Uncertainty: bootstrap 95% interval of the test F1
rng = np.random.default_rng(SEED)
y_test_arr, boot = y_test.to_numpy(), []
for _ in range(1000):
    idx = rng.integers(0, len(y_test_arr), len(y_test_arr))
    boot.append(f1_score(y_test_arr[idx], pred_test[idx]))
f1_ci = np.percentile(boot, [2.5, 97.5])
print(f"Test F1 95% bootstrap interval: {f1_ci[0]:.3f} - {f1_ci[1]:.3f}")

## 18. Save the Model for Streamlit

The artifact stores the **complete pipeline** (cleaning, feature engineering, encoding, selection, model, calibration), the threshold and the input schema. `app.py` passes raw customer fields to it. Because `add_features` lives in `churn_utils.py`, the app can unpickle it.

In [ ]:
artifact = {
    "model": final_model,
    "threshold": best_threshold,
    "model_name": final_name,
    "feature_set": best_fs,
    "calibration": best_label,
    "probabilities_calibrated": best_label != "uncalibrated",
    "input_columns": list(X_train.columns),
    "categories": {c: sorted(X_train[c].unique().tolist()) for c in X_train.select_dtypes(include=["object", "string"]).columns
                   if c != "TotalCharges"},
    "sklearn_version": sklearn.__version__,
}
os.makedirs(os.path.join(REPO_ROOT, "models"), exist_ok=True)
MODEL_PATH = os.path.join(REPO_ROOT, "models", "final_model.joblib")
joblib.dump(artifact, MODEL_PATH)

# Check: reload and score raw rows (also with a blank TotalCharges, as in the raw file)
loaded = joblib.load(MODEL_PATH)
same = np.allclose(loaded["model"].predict_proba(X_test)[:, 1], proba_test)
print("Reloaded model reproduces test probabilities:", same)
print("File size (MB):", round(os.path.getsize(MODEL_PATH) / 1e6, 2))

## 19. Final Conclusions and Limitations

The numbers are generated by the next cell, so the text always matches the executed run. Copy them into the README.

**Limitations (true regardless of the scores)**
- Fictional, single-snapshot data: no time dimension, so we cannot test performance on future customers, and some features (e.g. `tenure`) may partly reflect the churn outcome.
- One 80/20 split on ~7k customers: scores have uncertainty (see the bootstrap interval). Small differences between models are not meaningful.
- The threshold maximises F1 and ignores campaign costs; a real project should use cost-based thresholds.
- The model predicts association, not why customers leave; it should not be used to make real decisions about customers.

In [ ]:
selected_row = tuned[tuned["model"] == final_name + " (tuned)"].iloc[0]
print(f"Feature set: {best_fs} | model: {final_name} | reduced features: {use_reduced} | calibration: {best_label}")
print(f"Tuned params: {best_params}")
print(f"CV validation F1 (tuned model): {selected_row['val_f1']:.3f} +/- {selected_row['val_f1_std']:.3f}")
print(f"Chosen threshold: {best_threshold:.2f}")
print(f"TEST: F1 {test_metrics['f1']:.3f} | precision {test_metrics['precision']:.3f} | recall {test_metrics['recall']:.3f} "
      f"| AP {test_metrics['avg_precision']:.3f} | ROC-AUC {test_metrics['roc_auc']:.3f}")
print(f"Test F1 95% bootstrap interval: {f1_ci[0]:.3f} - {f1_ci[1]:.3f}")

results = {
    "dataset_shape": list(df_raw.shape),
    "train_churn_rate": float(y_train.mean()),
    "feature_set": best_fs,
    "final_model": final_name,
    "best_params": {k: (None if v is None else v) for k, v in best_params.items()},
    "reduced_features_used": bool(use_reduced),
    "calibration": best_label,
    "threshold": best_threshold,
    "cv_comparison_default": comparison.drop(columns="feature_set").round(4).to_dict(orient="records"),
    "cv_comparison_tuned": tuned.round(4).to_dict(orient="records"),
    "full_vs_reduced": [full_row, reduced_row],
    "test_metrics": test_metrics,
    "test_f1_bootstrap_95": f1_ci.tolist(),
    "sklearn_version": sklearn.__version__,
}
with open(os.path.join(REPO_ROOT, "models", "results_summary.json"), "w") as f:
    json.dump(results, f, indent=2, default=float)
print("Saved models/results_summary.json")